In [1]:
#Create the DataFrames
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer_id": [101, 102, 103, 104],
    "customer_name": ["Ali", "Sara", "Reza", "Mina"],
    "city": ["Tehran", "Shiraz", "Isfahan", "Tabriz"]
})

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004],
    "customer_id": [101, 102, 101, 105],
    "amount": [500, 750, 300, 900]
})

display(customers)
display(orders)

,customer_id,customer_name,city
0,101,Ali,Tehran
1,102,Sara,Shiraz
2,103,Reza,Isfahan
3,104,Mina,Tabriz


,order_id,customer_id,amount
0,1001,101,500
1,1002,102,750
2,1003,101,300
3,1004,105,900


In [2]:
#first merge :
merged = pd.merge(
    customers,
    orders,
    on='customer_id'
)
display(merged)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001,500
1,101,Ali,Tehran,1003,300
2,102,Sara,Shiraz,1002,750


Pandas matched the shared column.

In [5]:
inner = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='inner'
)
display(inner)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001,500
1,101,Ali,Tehran,1003,300
2,102,Sara,Shiraz,1002,750


In [6]:
left = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='left'
)
display(left)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001.0,500.0
1,101,Ali,Tehran,1003.0,300.0
2,102,Sara,Shiraz,1002.0,750.0
3,103,Reza,Isfahan,NaN,NaN
4,104,Mina,Tabriz,NaN,NaN


In [7]:
right = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='right'
)
display(right)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001,500
1,102,Sara,Shiraz,1002,750
2,101,Ali,Tehran,1003,300
3,105,NaN,NaN,1004,900


In [8]:
outer = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='outer'
)
display(outer)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001.0,500.0
1,101,Ali,Tehran,1003.0,300.0
2,102,Sara,Shiraz,1002.0,750.0
3,103,Reza,Isfahan,NaN,NaN
4,104,Mina,Tabriz,NaN,NaN
5,105,NaN,NaN,1004.0,900.0


In [9]:
#Handling different key names and duplicate column names
#Real datasets don't always use the same column name for the same information.
customers2 = pd.DataFrame({
    "customer_id": [101, 102, 103],
    "customer_name": ["Ali", "Sara", "Reza"]
})

orders2 = pd.DataFrame({
    "order_id": [1001, 1002, 1003],
    "client_id": [101, 102, 101],
    "amount": [500, 750, 300]
})

In [11]:
#Use left_on and right_on
merged = pd.merge(
    customers2,
    orders2,
    left_on='customer_id',
    right_on='client_id',
    how='inner'
)
display(merged)

,customer_id,customer_name,order_id,client_id,amount
0,101,Ali,1001,101,500
1,101,Ali,1003,101,300
2,102,Sara,1002,102,750


Because the column names differ, both key columns appear in the output. You can drop the redundant one if appropriate.

In [13]:
#What if both tables have columns with the same name?
customers3 = pd.DataFrame({
    "customer_id": [101, 102],
    "city": ["Tehran", "Shiraz"]
})

orders3 = pd.DataFrame({
    "customer_id": [101, 102],
    "city": ["Tehran Warehouse", "Shiraz Warehouse"]
})

In [15]:
result = pd.merge(
    customers3,
    orders3,
    on='customer_id',
    suffixes=['_customer','_order'],
    how='inner'
)
display(result)

,customer_id,city_customer,city_order
0,101,Tehran,Tehran Warehouse
1,102,Shiraz,Shiraz Warehouse


This is useful when you need to compare values from two sources.

In [18]:
#Validate your merge
merged = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='left',
    validate="one_to_many"
)
display(merged)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001.0,500.0
1,101,Ali,Tehran,1003.0,300.0
2,102,Sara,Shiraz,1002.0,750.0
3,103,Reza,Isfahan,NaN,NaN
4,104,Mina,Tabriz,NaN,NaN


In [20]:
#Mini project — Customer Sales Analysis
#Which customers placed orders, how much did they spend in total, and which customers haven't purchased anything?
result = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='left'
)
display(result)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001.0,500.0
1,101,Ali,Tehran,1003.0,300.0
2,102,Sara,Shiraz,1002.0,750.0
3,103,Reza,Isfahan,NaN,NaN
4,104,Mina,Tabriz,NaN,NaN


In [21]:
#Calculate total spending per customer
result_calculate = result.groupby(['customer_id','customer_name','city']).agg(
    total_amount = ('amount','sum'),
    number_of_orders = ('order_id','count')
)
display(result_calculate)

,,,total_amount,number_of_orders
customer_id,customer_name,city,,
101,Ali,Tehran,800.0,2
102,Sara,Shiraz,750.0,1
103,Reza,Isfahan,0.0,0
104,Mina,Tabriz,0.0,0


In [24]:
#Find customers who haven't purchased anything
no_purchase = result_calculate.loc[result_calculate['total_amount'] == 0]

In [25]:
display(no_purchase)

,,,total_amount,number_of_orders
customer_id,customer_name,city,,
103,Reza,Isfahan,0.0,0
104,Mina,Tabriz,0.0,0


In [26]:
#Find customers with multiple orders
mult_orders = result_calculate.loc[result_calculate['number_of_orders'] > 1]
display(mult_orders)

,,,total_amount,number_of_orders
customer_id,customer_name,city,,
101,Ali,Tehran,800.0,2


In [27]:
#Final exercises
#Merge customers and orders using an inner join.
merged1 = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='inner'
)
display(merged1)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001,500
1,101,Ali,Tehran,1003,300
2,102,Sara,Shiraz,1002,750


In [28]:
#Which customer ID appears in the orders table but not in the customers table?
merged2 = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='outer'
)
display(merged2)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001.0,500.0
1,101,Ali,Tehran,1003.0,300.0
2,102,Sara,Shiraz,1002.0,750.0
3,103,Reza,Isfahan,NaN,NaN
4,104,Mina,Tabriz,NaN,NaN
5,105,NaN,NaN,1004.0,900.0


In [29]:
#filter for customers whose total spending exceeds 700.
result_calculate.loc[result_calculate['total_amount'] > 700]

,,,total_amount,number_of_orders
customer_id,customer_name,city,,
101,Ali,Tehran,800.0,2
102,Sara,Shiraz,750.0,1
